# NuCLS · Experimento v2 en GPU
**Andrés Duarte — Deep Learning**

Ejecución de softmaxCNN-BN adaptada, DenseNet121 y ResNet50 desde cero. Se mantienen la partición por pacientes y centros, la búsqueda de hiperparámetros y la regla de planificación de corridas fijadas en el protocolo.

## Preparación
1. Subir `NuCLS_V2_DATOS.zip` a la raíz de **Mi unidad** de Google Drive.
2. En Colab seleccionar **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU**.
3. Ejecutar las celdas en orden. Las imágenes se leen desde el disco temporal de Colab; los checkpoints y resultados se guardan en Drive.

La ejecución incluye 12 ajustes de búsqueda, 15 del piloto y al menos 15 finales. Puede requerir varias horas o sesiones; no se estima duración sin medir la GPU. Se guardan checkpoints por época. La planificación se detiene si requiere más de 50 corridas finales por modelo, sin evaluar la reserva.

Este cuaderno se entrega sin salidas de entrenamiento: los resultados deben proceder de su ejecución real. El entrenamiento no sustituye el cierre de la revisión bibliográfica ni el análisis agrupado de los seis pacientes de reserva.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import sys, subprocess, importlib.util
for module, package in [('numpy','numpy'),('scipy','scipy'),('PIL','Pillow')]:
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable,'-m','pip','install',package],check=True)
import torch, torchvision, numpy, scipy
if not torch.cuda.is_available():
    raise RuntimeError('Seleccionar GPU en el menú Entorno de ejecución antes de continuar.')
print('GPU:',torch.cuda.get_device_name(0))
print('PyTorch:',torch.__version__,'Torchvision:',torchvision.__version__)
print('NumPy:',numpy.__version__,'SciPy:',scipy.__version__)


Mounted at /content/drive
GPU: Tesla T4
PyTorch: 2.14.1+cu130 Torchvision: 0.29.1+cu130
NumPy: 2.4.6 SciPy: 1.16.3


## Datos y código fijados
Se verifica el manifiesto y la integridad binaria de los archivos durante la extracción, sin calcular características ni predicciones de la reserva. Los archivos originales en Drive permanecen intactos.

In [ ]:
import hashlib, csv, zipfile, urllib.request, shutil
COMMIT='6d672bcb13a33536f9e1c48d002c6f7c513d969e'
EXPECTED='623adf36520751addc324e44cd1b89d95e47842e53cda494914c0248a8be86d4'
ZIP_DRIVE=Path('/content/drive/MyDrive/NuCLS_V2_DATOS.zip')
DATOS=Path('/content/nucls_v2_datos')
CODIGO=Path('/content/nucls_v2_codigo/modelos')
SALIDA=Path('/content/drive/MyDrive/NuCLS_V2_RESULTADOS_6d672bcb')
if not ZIP_DRIVE.is_file():
    raise FileNotFoundError('Subir NuCLS_V2_DATOS.zip a la raíz de Mi unidad.')
local_zip=Path('/content/NuCLS_V2_DATOS.zip')
shutil.copyfile(ZIP_DRIVE, local_zip)
DATOS.mkdir(exist_ok=True); CODIGO.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(local_zip) as z:
    raw=z.read('manifiesto.csv')
    if hashlib.sha256(raw).hexdigest()!=EXPECTED:
        raise ValueError('El manifiesto no corresponde al conjunto v2 verificado.')
    rows=list(csv.DictReader(raw.decode().splitlines()))
    if len(rows)!=17557: raise ValueError('Número de recortes inesperado.')
    for i,row in enumerate(rows,1):
        target=(DATOS/row['archivo']).resolve()
        if not target.is_relative_to(DATOS.resolve()): raise ValueError('Ruta no válida.')
        data=z.read(row['archivo'])
        if hashlib.sha256(data).hexdigest()!=row['sha_archivo']:
            raise ValueError('Archivo diferente: '+row['archivo'])
        target.parent.mkdir(parents=True,exist_ok=True);target.write_bytes(data)
        if i%2000==0: print('Recortes verificados:',i,flush=True)
    (DATOS/'manifiesto.csv').write_bytes(raw)
for name in ['candidatos_v2.py','core_v2.py','experimento_v2.py']:
    url=f'https://raw.githubusercontent.com/aduartech81/NuCLS_DeepLearning/{COMMIT}/modelos/{name}'
    with urllib.request.urlopen(url,timeout=60) as response: data=response.read()
    compile(data,name,'exec');(CODIGO/name).write_bytes(data)
print('Recortes disponibles:',len(rows))
print('Resultados y checkpoints:',SALIDA)


Recortes verificados: 2000
Recortes verificados: 4000
Recortes verificados: 6000
Recortes verificados: 8000
Recortes verificados: 10000
Recortes verificados: 12000
Recortes verificados: 14000
Recortes verificados: 16000
Recortes disponibles: 17557
Resultados y checkpoints: /content/drive/MyDrive/NuCLS_V2_RESULTADOS_6d672bcb


## Entrenamiento y evaluación
La selección se realiza exclusivamente con validación. Antes de evaluar la reserva se fijan el modelo de referencia, las semillas y las huellas de todos los checkpoints finales.

Si Colab se desconecta, se puede volver a ejecutar el cuaderno con la misma carpeta de salida. El motor comprueba configuración, código y entorno antes de reanudar; si detecta cambios, se detiene para evitar mezclar ejecuciones. No debe borrarse la carpeta de resultados para resolver ese mensaje. Drive necesita espacio para varios checkpoints por corrida.

In [ ]:
import subprocess
import sys

try:
    subprocess.run([
        sys.executable, str(CODIGO/'experimento_v2.py'),
        '--datos', str(DATOS), '--salida', str(SALIDA), '--device', 'cuda'
    ], check=True, capture_output=True, text=True)
except subprocess.CalledProcessError as e:
    print("--- ERROR DURANTE LA EJECUCIÓN DEL SCRIPT ---")
    print("SALIDA ESTÁNDAR (STDOUT):")
    print(e.stdout)
    print("\nSALIDA DE ERROR (STDERR):")
    print(e.stderr)
    raise e

--- ERROR DURANTE LA EJECUCIÓN DEL SCRIPT ---
SALIDA ESTÁNDAR (STDOUT):
Verificando entrenamiento y validación; no se abren imágenes de reserva.
Verificando train 0 / 14539
Verificando train 2000 / 14539
Verificando train 4000 / 14539
Verificando train 6000 / 14539
Verificando train 8000 / 14539
Verificando train 10000 / 14539
Verificando train 12000 / 14539
Verificando train 14000 / 14539
Verificando validation 0 / 2049
Verificando validation 2000 / 2049


SALIDA DE ERROR (STDERR):
Traceback (most recent call last):
  File "/content/nucls_v2_codigo/modelos/experimento_v2.py", line 103, in <module>
    if __name__=='__main__':main()
                            ~~~~^^
  File "/content/nucls_v2_codigo/modelos/experimento_v2.py", line 49, in main
    run_one(model,4001,groups,args,device);q=result(args.salida,model,4001);search.append({'modelo':model,'configuracion':i,**hp,'macro_f1':q['validacion']['macro_f1']})
    ~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/nucls_v2_codigo/

CalledProcessError: Command '['/usr/bin/python3', '/content/nucls_v2_codigo/modelos/experimento_v2.py', '--datos', '/content/nucls_v2_datos', '--salida', '/content/drive/MyDrive/NuCLS_V2_RESULTADOS_6d672bcb', '--device', 'cuda']' returned non-zero exit status 1.

In [13]:
from pathlib import Path
import urllib.request
import subprocess
import sys

revision = "bd48a2ca8b50a883912a9fcbcb3435da8aec372a"
url = (
    "https://raw.githubusercontent.com/"
    f"aduartech81/NuCLS_DeepLearning/{revision}/"
    "modelos/candidatos_v2.py"
)

with urllib.request.urlopen(url, timeout=60) as respuesta:
    codigo = respuesta.read()

compile(codigo, "candidatos_v2.py", "exec")
(CODIGO / "candidatos_v2.py").write_bytes(codigo)

# Carpeta nueva para conservar el intento anterior
# y registrar correctamente la modificación del código.
SALIDA = Path(
    "/content/drive/MyDrive/NuCLS_V2_RESULTADOS_pool_corregido"
)

proceso = subprocess.Popen(
    [
        sys.executable, "-u",
        str(CODIGO / "experimento_v2.py"),
        "--datos", str(DATOS),
        "--salida", str(SALIDA),
        "--device", "cuda"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

for linea in proceso.stdout:
    print(linea, end="", flush=True)

print("\nCódigo de salida:", proceso.wait())

Verificando entrenamiento y validación; no se abren imágenes de reserva.
Verificando train 0 / 14539
Verificando train 2000 / 14539
Verificando train 4000 / 14539
Verificando train 6000 / 14539
Verificando train 8000 / 14539
Verificando train 10000 / 14539
Verificando train 12000 / 14539
Verificando train 14000 / 14539
Verificando validation 0 / 2049
Verificando validation 2000 / 2049
softmaxcnn_bn 4001 época 1 lote 100 / 455
softmaxcnn_bn 4001 época 1 lote 200 / 455
softmaxcnn_bn 4001 época 1 lote 300 / 455
softmaxcnn_bn 4001 época 1 lote 400 / 455
softmaxcnn_bn semilla=4001 época=1 loss=0.7213 val_F1=0.6506 tiempo=54.2s
softmaxcnn_bn 4001 época 2 lote 100 / 455
softmaxcnn_bn 4001 época 2 lote 200 / 455
softmaxcnn_bn 4001 época 2 lote 300 / 455
softmaxcnn_bn 4001 época 2 lote 400 / 455
softmaxcnn_bn semilla=4001 época=2 loss=0.5951 val_F1=0.6895 tiempo=53.2s
softmaxcnn_bn 4001 época 3 lote 100 / 455
softmaxcnn_bn 4001 época 3 lote 200 / 455
softmaxcnn_bn 4001 época 3 lote 300 / 455
so

KeyboardInterrupt: 

## Evidencia de ejecución
El ZIP contiene métricas, predicciones y registros; los pesos se conservan por separado en Drive. El estado distingue un experimento completado de una interrupción por el límite de corridas. Una ejecución completa requiere posteriormente interpretar los resultados e incorporarlos al informe y al dashboard.

In [15]:
import json
estado=SALIDA/'estado.json'
plan=SALIDA/'plan_corridas.json'
if estado.exists():
    print(json.dumps(json.loads(estado.read_text()),indent=2,ensure_ascii=False))
elif plan.exists():
    print(json.dumps(json.loads(plan.read_text()),indent=2,ensure_ascii=False))
else:
    print('Ejecución incompleta. Revisar el mensaje anterior; los checkpoints permanecen en Drive.')
for path in sorted(SALIDA.parent.glob(SALIDA.name+'*.zip')):
    print('ZIP de resultados:',path)


{
  "comparaciones": [
    {
      "A": "softmaxcnn_bn",
      "B": "densenet121",
      "sd_diferencias": 0.01719576941508776,
      "R_requerido": 11
    },
    {
      "A": "softmaxcnn_bn",
      "B": "resnet50",
      "sd_diferencias": 0.02663393267102482,
      "R_requerido": 22
    },
    {
      "A": "densenet121",
      "B": "resnet50",
      "sd_diferencias": 0.018123600408532583,
      "R_requerido": 12
    }
  ],
  "R_por_modelo": 22,
  "regla": "Máximo R para detectar delta .02 en tres diferencias pareadas con alpha .05/3 y potencia .8; mínimo cinco",
  "limitacion": "Estimación basada en solo cinco pares; no incluye variabilidad entre pacientes",
  "aprobado_por_limite_computacional": true
}


In [16]:
from google.colab import files
from pathlib import Path

subidos = files.upload()
nombre = "cerrar_nucls_11.py"
assert nombre in subidos, "Selecciona el archivo cerrar_nucls_11.py"
CIERRE = Path("/content/cerrar_nucls_11.py")
CIERRE.write_bytes(subidos[nombre])
print("Archivo de cierre preparado.")

Saving cerrar_nucls_11.py to cerrar_nucls_11.py
Archivo de cierre preparado.


In [17]:
import json
import subprocess

ORIGEN = Path(
    "/content/drive/MyDrive/NuCLS_V2_RESULTADOS_pool_corregido"
)
modelos = ["softmaxcnn_bn", "densenet121", "resnet50"]

for modelo in modelos:
    for semilla in range(6101, 6112):
        carpeta = ORIGEN / "final" / f"{modelo}_{semilla}"
        resultado = json.loads(
            (carpeta / "resultado.json").read_text()
        )
        assert (
            resultado["modelo"],
            resultado["semilla"],
            resultado["estado"]
        ) == (
            modelo, semilla, "FASE_COMPLETADA_VALIDACION"
        ), f"Resultado incompleto: {modelo}_{semilla}"
        assert (carpeta / "mejor.pt").stat().st_size > 0

print("VERIFICADO: 11 corridas completas por modelo, 33 en total.")

# Detener la celda puede dejar vivo el subproceso.
# Solo se termina el proceso original identificado.
if "proceso" in globals() and proceso.poll() is None:
    argumentos = proceso.args
    assert isinstance(argumentos, (list, tuple))
    assert str(Path(CODIGO) / "experimento_v2.py") in argumentos
    assert "--salida" in argumentos
    posicion = argumentos.index("--salida")
    assert Path(argumentos[posicion + 1]).resolve() == ORIGEN.resolve()

    proceso.terminate()
    try:
        proceso.wait(timeout=30)
    except subprocess.TimeoutExpired:
        raise RuntimeError(
            "El entrenamiento sigue activo. No ejecutes la evaluación."
        )

# Comprobar que no quede otro proceso de entrenamiento.
activos = []
for carpeta in Path("/proc").iterdir():
    if not carpeta.name.isdigit():
        continue
    try:
        argumentos = (carpeta / "cmdline").read_bytes().split(b"\0")
    except (FileNotFoundError, PermissionError, ProcessLookupError):
        continue
    if any(
        argumento.decode(errors="replace").endswith("/experimento_v2.py")
        for argumento in argumentos
    ):
        activos.append(carpeta.name)

assert not activos, f"Entrenamiento todavía activo: {activos}"
print("LISTO PARA EVALUAR: entrenamiento detenido; archivos conservados.")

VERIFICADO: 11 corridas completas por modelo, 33 en total.
LISTO PARA EVALUAR: entrenamiento detenido; archivos conservados.


In [18]:
import sys

subprocess.run(
    [
        sys.executable, "-u", str(CIERRE),
        "--codigo", str(CODIGO),
        "--datos", str(DATOS),
        "--origen", str(ORIGEN),
        "--evaluar",
    ],
    check=True,
)

CompletedProcess(args=['/usr/bin/python3', '-u', '/content/cerrar_nucls_11.py', '--codigo', '/content/nucls_v2_codigo/modelos', '--datos', '/content/nucls_v2_datos', '--origen', '/content/drive/MyDrive/NuCLS_V2_RESULTADOS_pool_corregido', '--evaluar'], returncode=0)

In [19]:
import json
import pandas as pd
from pathlib import Path

CARPETA_CIERRE = Path(
    "/content/drive/MyDrive/"
    "NuCLS_V2_RESULTADOS_pool_corregido_CIERRE_11"
)

print(json.dumps(
    json.loads((CARPETA_CIERRE / "estado.json").read_text()),
    indent=2, ensure_ascii=False
))
display(pd.read_csv(CARPETA_CIERRE / "resumen_experimento.csv"))

{
  "estado": "CIERRE_11_COMPLETADO_PLAN_22_PENDIENTE",
  "baseline": "resnet50",
  "R": 11,
  "R_plan_original": 22,
  "advertencia": "IC por semillas condicionados a validación. Seis pacientes de reserva requieren análisis agrupado adicional; no son 969 pacientes."
}


,modelo,R,baseline,macro_f1_val_media,macro_f1_val_sd,IC95_semillas_inf,IC95_semillas_sup,macro_f1_reserva_media,macro_f1_reserva_sd
0,softmaxcnn_bn,11,False,0.756766,0.025475,0.739652,0.773880,0.454185,0.071708
1,densenet121,11,False,0.778256,0.025384,0.761203,0.795309,0.605226,0.111180
2,resnet50,11,True,0.793404,0.017341,0.781754,0.805054,0.637771,0.051807


In [20]:
from pathlib import Path
from google.colab import files

raiz = Path("/content/drive/MyDrive")
zips = sorted(
    raiz.glob("*CIERRE_11*.zip"),
    key=lambda p: p.stat().st_mtime,
    reverse=True
)

if zips:
    print("ZIP del cierre:", zips[0])
    files.download(str(zips[0]))
else:
    print("Copia aquí la línea que empieza por 'CIERRE DE 11 COMPLETADO'.")

ZIP del cierre: /content/drive/MyDrive/NuCLS_V2_RESULTADOS_pool_corregido_CIERRE_11_RESULTADOS.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>